# SLIIT IT3051 - EV Battery PHM Dual-Task System
## Week 2 - Task 2: Linear Baselines, Decision Threshold Tuning & Master Leaderboard
### **Assigned Member: Person D**

---
### Overview & Objectives:
Establish classification baselines, evaluate cost-sensitive linear models, perform decision threshold tuning for safety, and compile the Master Evaluation Leaderboard.


### 0. Environment Setup & Data Partition Loading
Execute this cell to load the verified preprocessed matrices.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

def load_data():
    dataset_path = 'ev battery_failure  Dataset.csv'
    for p in [dataset_path, f'../{dataset_path}', f'../../{dataset_path}']:
        if os.path.exists(p):
            dataset_path = p
            break
            
    df = pd.read_csv(dataset_path)
    
    # 7 Domain Features
    df['temperature_spread'] = df['cell_temperature_max'] - df['cell_temperature_avg']
    df['efficiency_gap'] = (df['charge_efficiency'] - df['discharge_efficiency']).abs()
    df['health_loss_interaction'] = (df['battery_health_percent'] * df['capacity_loss_percent']) / 100.0
    df['resistance_per_1000_cycles'] = (df['internal_resistance'] / (df['cycle_count'] + 1.0)) * 1000.0
    df['c_rate_proxy'] = df['average_charge_power_kw'] / (df['battery_capacity_kwh'] + 1e-5)
    df['cell_voltage_spread'] = df['cell_voltage_std'] / (df['cell_voltage_avg'] + 1e-5)
    df['stress_index'] = df['aggressive_acceleration_score'] * df['hard_braking_score']
    
    id_cols = ['vehicle_id', 'battery_serial']
    target_rul = 'predicted_remaining_life_cycles'
    target_fail = 'battery_failure'
    
    excluded = id_cols + [target_rul, target_fail]
    features = [c for c in df.columns if c not in excluded]
    
    num_cols = df[features].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    
    # Task 1 Splits (RUL)
    df_t1 = df[df[target_rul].notnull()].copy()
    X1_tr, X1_te, y1_train, y1_test = train_test_split(df_t1[features], df_t1[target_rul], test_size=0.20, random_state=42)
    ct1 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X1_train = ct1.fit_transform(X1_tr)
    X1_test = ct1.transform(X1_te)
    
    # Task 2 Splits (Failure)
    X2_tr, X2_te, y2_train, y2_test = train_test_split(df[features], df[target_fail], test_size=0.20, random_state=42, stratify=df[target_fail])
    ct2 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X2_train = ct2.fit_transform(X2_tr)
    X2_test = ct2.transform(X2_te)
    
    feature_names = ct1.get_feature_names_out()
    return (X1_train, X1_test, y1_train, y1_test), (X2_train, X2_test, y2_train, y2_test), feature_names

print("Data Loader initialized successfully!")



In [ ]:
_, (X2_train, X2_test, y2_train, y2_test), feature_names = load_data()
print(f'Task 2 Train Shape: {X2_train.shape} | Test Shape: {X2_test.shape}')


### Task 1: Classification Baselines: Dummy Classifier & Unweighted Logistic Regression
Train a Dummy Classifier (strategy='most_frequent') and an unweighted standard Logistic Regression model. Examine the Accuracy Paradox.


In [ ]:
# TODO: Import DummyClassifier and LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

# TODO: Fit DummyClassifier(strategy='most_frequent') on X2_train, y2_train

# TODO: Fit unweighted LogisticRegression(max_iter=1000) on X2_train, y2_train

# TODO: Compare Accuracy vs Recall on test set



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- What accuracy does the Dummy Classifier achieve, and what is its Recall?
- Why does unweighted Logistic Regression have high accuracy but poor recall in this imbalanced dataset?


### Task 2: Cost-Sensitive Linear Classification
Train Logistic Regression with class_weight='balanced' and compare against the unweighted baseline. Also evaluate a linear SGDClassifier with loss='log_loss'.


In [ ]:
# TODO: Train LogisticRegression(class_weight='balanced', max_iter=1000)

# TODO: Train SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42)

# TODO: Compute confusion matrix and evaluate Precision, Recall, and PR-AUC



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- How does applying class_weight='balanced' change the decision boundary?
- What is the reduction in False Negatives compared to the unweighted model?


### Task 3: Decision Threshold Tuning (Automotive Safety Calibration)
Extract predicted probabilities using model.predict_proba(X2_test)[:, 1]. Plot the Precision-Recall curve across threshold cutoffs (0.0 to 1.0). Find an optimal threshold (e.g. tau = 0.30) that maximizes Recall while keeping Precision acceptable.


In [ ]:
# TODO: Get predicted probabilities for the positive class (failure)

# TODO: Plot Precision and Recall as a function of the probability threshold

# TODO: Evaluate confusion matrix at default cutoff (0.50) vs tuned safety cutoff (e.g. 0.30)

# TODO: Calculate the number of additional battery failures detected



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Why is the default 0.50 threshold dangerous in safety-critical EV battery prognostics?
- What threshold value did you select, and how many battery failures does it successfully prevent?


### Task 4: Master Comparison Leaderboards Compilation
Compile the unified master comparison tables for both Task 1 (Regression) and Task 2 (Classification) synthesizing results from all 4 teammates.


In [ ]:
# TODO: Create Task 1 Leaderboard DataFrame (Models: Dummy, Linear, Ridge, Lasso, Random Forest, XGBoost)
# Columns: Model, Type, Author, Test R2, Test RMSE, Test MAE

# TODO: Create Task 2 Leaderboard DataFrame (Models: Dummy, Logistic, Balanced Logistic, Balanced RF, Tuned Threshold)
# Columns: Model, Imbalance Strategy, Author, Accuracy, Precision, Recall, PR-AUC, ROC-AUC

# TODO: Display both leaderboards and highlight the champion model for each task



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Which model is recommended as Champion for Task 1 (RUL Regression)? Justify with R2 and RMSE.
- Which model is recommended as Champion for Task 2 (Failure Classification)? Justify with Recall and PR-AUC.
